In [3]:
import numpy as np
import pandas as pd 
import statistics 

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import SGDClassifier
from sklearn.kernel_approximation import Nystroem

In [7]:
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.under_sampling import TomekLinks
from imblearn.pipeline import Pipeline

In [8]:
import os 
from PIL import Image
import cv2
from pathlib import Path

In [10]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [11]:
TARGET_SIZE = (64, 64)
path = Path(r'C:\Users\Acer\plantvillage-dataset\color')

In [59]:
"""def load_data(directories):
    X= []
    y = []
    for direc in directories:
        direc = os.path.join(path,Path(direc))
        #print(direc)
        label = Path(direc).parts[-1]
        #print(label)
        for image_path in Path(direc).glob('*.*'):
            image = cv2.imread(image_path)
            image = cv2.resize(image, TARGET_SIZE)
            image_array = image.flatten()
            X.append(image_array)
            y.append(label)
    return X,y """

from joblib import Parallel, delayed
from skimage.feature import hog

cv2.setUseOptimized(True)
cv2.setNumThreads(0)

def process_image(image_path, label):
    image = cv2.imread(image_path)
    image = cv2.resize(image, TARGET_SIZE)
    image = image.astype(np.float32) / 255.0
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    image_hog = hog(gray)
    return image_hog, label

def load_data_parallel(directories):
    tasks = []
    for direc in directories:
        direc_path = os.path.join(path, direc)
        label = direc
        for img_path in Path(direc_path).glob('*'):
            tasks.append((img_path, label))

    results = Parallel(n_jobs=-1, backend='threading')(
        delayed(process_image)(p, l) for p, l in tasks
    )

    X, y = zip(*results)
    return np.array(X), np.array(y)


In [61]:
directories = [x for x in os.listdir(path)]
X,y = load_data_parallel(directories)

In [63]:
X

array([[0.04853482, 0.0446101 , 0.01685505, ..., 0.01436356, 0.01543768,
        0.04801571],
       [0.06212913, 0.04062561, 0.06524501, ..., 0.01731247, 0.03509339,
        0.02271868],
       [0.03841615, 0.01244771, 0.01054159, ..., 0.00459186, 0.02757033,
        0.02681815],
       ...,
       [0.08060958, 0.05021351, 0.05134483, ..., 0.03521293, 0.03103832,
        0.04029012],
       [0.1264922 , 0.04738848, 0.07977526, ..., 0.02119502, 0.00579749,
        0.0093724 ],
       [0.06223692, 0.02856035, 0.02635355, ..., 0.0241724 , 0.00794495,
        0.00567472]], dtype=float32)

In [65]:
y

array(['Apple___Apple_scab', 'Apple___Apple_scab', 'Apple___Apple_scab',
       ..., 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
       'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
       'Tomato___Tomato_Yellow_Leaf_Curl_Virus'], dtype='<U50')

In [22]:
pca = PCA(n_components=0.95)

In [69]:
X_scaled = scaler.fit_transform(X)

In [71]:
X_scaled

array([[-0.7984938 , -0.029999  , -0.7786606 , ..., -0.6951931 ,
        -0.69532275,  0.23175935],
       [-0.5222798 , -0.13981035,  0.42334968, ..., -0.62625647,
        -0.18064956, -0.43527564],
       [-1.0040884 , -0.9163837 , -0.93548745, ..., -0.92362654,
        -0.37763652, -0.32718047],
       ...,
       [-0.1467876 ,  0.12442908,  0.07806818, ..., -0.20779635,
        -0.28682923,  0.02805012],
       [ 0.7854714 ,  0.04657223,  0.78428215, ..., -0.5354938 ,
        -0.94774556, -0.78719187],
       [-0.5200896 , -0.47232473, -0.5427171 , ..., -0.46589145,
        -0.89151543, -0.88469267]], dtype=float32)

In [73]:
X_pca = pca.fit_transform(X_scaled)

In [77]:
X_pca.shape

(54305, 455)

In [79]:
X_train, X_test , y_train, y_test = train_test_split(X_pca, y , stratify = y , test_size = 0.2)
svm = SVC()

In [81]:
svm.fit(X_train, y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [83]:
y_pred = svm.predict(X_test)

In [87]:
print(classification_report(y_test,y_pred))

                                                    precision    recall  f1-score   support

                                Apple___Apple_scab       0.69      0.38      0.49       126
                                 Apple___Black_rot       0.71      0.69      0.70       124
                          Apple___Cedar_apple_rust       0.67      0.15      0.24        55
                                   Apple___healthy       0.54      0.59      0.56       329
                               Blueberry___healthy       0.69      0.69      0.69       300
          Cherry_(including_sour)___Powdery_mildew       0.86      0.86      0.86       210
                 Cherry_(including_sour)___healthy       0.92      0.81      0.86       171
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot       0.75      0.49      0.59       103
                       Corn_(maize)___Common_rust_       0.93      0.94      0.94       239
               Corn_(maize)___Northern_Leaf_Blight       0.70      0.79      0.

In [91]:
svm_linear = LinearSVC()

In [93]:
svm_linear.fit(X_train,y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,None


In [94]:
y_pred_linear = svm_linear.predict(X_test)
print(classification_report(y_test, y_pred_linear))

                                                    precision    recall  f1-score   support

                                Apple___Apple_scab       0.38      0.20      0.26       126
                                 Apple___Black_rot       0.52      0.61      0.57       124
                          Apple___Cedar_apple_rust       0.17      0.13      0.14        55
                                   Apple___healthy       0.48      0.29      0.36       329
                               Blueberry___healthy       0.49      0.44      0.46       300
          Cherry_(including_sour)___Powdery_mildew       0.70      0.75      0.72       210
                 Cherry_(including_sour)___healthy       0.64      0.61      0.62       171
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot       0.53      0.48      0.50       103
                       Corn_(maize)___Common_rust_       0.80      0.88      0.84       239
               Corn_(maize)___Northern_Leaf_Blight       0.61      0.61      0.

In [111]:
class_weight_vect = compute_class_weight(class_weight="balanced", classes=np.unique(y), y=y)
dict(zip(np.unique(y),class_weight_vect))

{'Apple___Apple_scab': 2.268379281537176,
 'Apple___Black_rot': 2.3012543435884396,
 'Apple___Cedar_apple_rust': 5.196650717703349,
 'Apple___healthy': 0.8687410014397696,
 'Blueberry___healthy': 0.9514506973158595,
 'Cherry_(including_sour)___Powdery_mildew': 1.358440064038423,
 'Cherry_(including_sour)___healthy': 1.6733945519536546,
 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 2.7857289422386375,
 'Corn_(maize)___Common_rust_': 1.1988917343694807,
 'Corn_(maize)___Northern_Leaf_Blight': 1.4508415709324072,
 'Corn_(maize)___healthy': 1.229844188785216,
 'Grape___Black_rot': 1.2110838537020518,
 'Grape___Esca_(Black_Measles)': 1.0333181108954599,
 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)': 1.328140285658384,
 'Grape___healthy': 3.378437227821326,
 'Orange___Haunglongbing_(Citrus_greening)': 0.25950226028117324,
 'Peach___Bacterial_spot': 0.6221501729945238,
 'Peach___healthy': 3.9696637426900585,
 'Pepper,_bell___Bacterial_spot': 1.433379084622288,
 'Pepper,_bell___health

In [47]:
svm_balanced = SVC(class_weight = "balanced")

In [107]:
svm_balanced.fit(X_train,y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,'balanced'
,verbose,False


In [108]:
y_pred_balanced = svm_balanced.predict(X_test)
print(classification_report(y_test, y_pred_balanced))

                                                    precision    recall  f1-score   support

                                Apple___Apple_scab       0.56      0.50      0.53       126
                                 Apple___Black_rot       0.65      0.77      0.71       124
                          Apple___Cedar_apple_rust       0.60      0.27      0.38        55
                                   Apple___healthy       0.51      0.60      0.55       329
                               Blueberry___healthy       0.62      0.76      0.68       300
          Cherry_(including_sour)___Powdery_mildew       0.83      0.89      0.86       210
                 Cherry_(including_sour)___healthy       0.90      0.85      0.87       171
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot       0.65      0.58      0.61       103
                       Corn_(maize)___Common_rust_       0.91      0.94      0.93       239
               Corn_(maize)___Northern_Leaf_Blight       0.71      0.73      0.

In [11]:
X = np.load("data_hog_PVD.npy")
y = np.load("labels_PVD.npy")

In [15]:
X_resampled, y_resampled = BorderlineSMOTE().fit_resample(X, y)

In [18]:
np.save("data_hog_SMOTE_PVD.npy", X_resampled)
np.save("labels_resampled_PVD.npy", y_resampled)

In [16]:
X.shape

(54305, 2916)

In [17]:
X_resampled.shape

(209266, 2916)

In [19]:
X_scaled_resampled = scaler.fit_transform(X_resampled)

In [24]:
X_pca_resampled = pca.fit_transform(X_scaled_resampled)
np.save("data_hog_SMOTE_PCA_PVD.npy", X_pca_resampled)

In [13]:
X_pca_resampled = np.load("data_hog_SMOTE_PCA_PVD.npy")
y_resampled = np.load("labels_resampled_PVD.npy")
X_train_balanced , X_test_balanced, y_train_balanced, y_test_balanced  = train_test_split(X_pca_resampled, y_resampled, stratify =y_resampled, test_size=0.2)
svm_linear_balanced = LinearSVC(class_weight="balanced")
sgd = SGDClassifier(loss = "hinge", n_jobs = -1, class_weight = "balanced")

In [24]:
X_train_balanced.shape

(167412, 443)

In [39]:
svm_linear_balanced.fit(X_train_balanced ,y_train_balanced)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,verbose,0
,random_state,None


In [43]:
y_pred_resampled = svm_linear_balanced.predict(X_test_balanced)

In [45]:
print(classification_report(y_test_balanced, y_pred_resampled))

                                                    precision    recall  f1-score   support

                                Apple___Apple_scab       0.71      0.79      0.75      1102
                                 Apple___Black_rot       0.84      0.90      0.87      1102
                          Apple___Cedar_apple_rust       0.87      0.96      0.91      1101
                                   Apple___healthy       0.62      0.48      0.54      1101
                               Blueberry___healthy       0.68      0.66      0.67      1102
          Cherry_(including_sour)___Powdery_mildew       0.83      0.91      0.87      1101
                 Cherry_(including_sour)___healthy       0.87      0.93      0.90      1101
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot       0.83      0.87      0.85      1101
                       Corn_(maize)___Common_rust_       0.95      0.98      0.97      1102
               Corn_(maize)___Northern_Leaf_Blight       0.79      0.75      0.

In [41]:
sgd.fit(X_train_balanced ,y_train_balanced)

,loss,'hinge'
,penalty,'l2'
,alpha,0.0001
,l1_ratio,0.15
,fit_intercept,True
,max_iter,1000
,tol,0.001
,shuffle,True
,verbose,0
,epsilon,0.1
,n_jobs,-1


In [42]:
y_pred_sgd = sgd.predict(X_test_balanced)
print(classification_report(y_test_balanced, y_pred_sgd))

                                                    precision    recall  f1-score   support

                                Apple___Apple_scab       0.66      0.74      0.70      1101
                                 Apple___Black_rot       0.76      0.86      0.81      1102
                          Apple___Cedar_apple_rust       0.81      0.90      0.85      1101
                                   Apple___healthy       0.52      0.44      0.48      1101
                               Blueberry___healthy       0.62      0.61      0.62      1102
          Cherry_(including_sour)___Powdery_mildew       0.75      0.79      0.77      1102
                 Cherry_(including_sour)___healthy       0.86      0.87      0.86      1101
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot       0.83      0.81      0.82      1101
                       Corn_(maize)___Common_rust_       0.96      0.93      0.94      1102
               Corn_(maize)___Northern_Leaf_Blight       0.76      0.72      0.

In [38]:
pipeline = Pipeline([
    ('nystrom', Nystroem(
        kernel='rbf',
        gamma=1e-4,
        n_components=2000,
        random_state=42
    )),
    ('clf', SGDClassifier(
        loss="hinge",
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    ))
])

pipeline.fit(X_train_balanced, y_train_balanced)

,steps,"[('nystrom', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,kernel,'rbf'
,gamma,0.0001
,coef0,None
,degree,None
,kernel_params,None
,n_components,2000
,random_state,42


In [39]:
y_pred_resampled = pipeline.predict(X_test_balanced)
print(classification_report(y_test_balanced, y_pred_resampled))

                                                    precision    recall  f1-score   support

                                Apple___Apple_scab       0.61      0.81      0.70      1101
                                 Apple___Black_rot       0.73      0.86      0.79      1101
                          Apple___Cedar_apple_rust       0.80      0.87      0.83      1101
                                   Apple___healthy       0.64      0.33      0.44      1102
                               Blueberry___healthy       0.81      0.46      0.59      1101
          Cherry_(including_sour)___Powdery_mildew       0.73      0.84      0.78      1101
                 Cherry_(including_sour)___healthy       0.78      0.89      0.83      1102
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot       0.66      0.84      0.74      1102
                       Corn_(maize)___Common_rust_       0.81      0.97      0.88      1101
               Corn_(maize)___Northern_Leaf_Blight       0.89      0.35      0.

In [54]:
randomforest = RandomForestClassifier(n_estimators = 80 , max_depth = 12 , n_jobs = -1 , class_weight = "balanced")

In [56]:
randomforest.fit(X_train_balanced, y_train_balanced)

,n_estimators,80
,criterion,'gini'
,max_depth,12
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [57]:
y_pred_rf = randomforest.predict(X_test_balanced)
print(classification_report(y_test_balanced, y_pred_rf))

                                                    precision    recall  f1-score   support

                                Apple___Apple_scab       0.91      0.84      0.87      1101
                                 Apple___Black_rot       0.80      0.98      0.88      1101
                          Apple___Cedar_apple_rust       0.90      0.96      0.93      1101
                                   Apple___healthy       0.68      0.59      0.63      1102
                               Blueberry___healthy       0.84      0.70      0.76      1101
          Cherry_(including_sour)___Powdery_mildew       0.85      0.94      0.89      1101
                 Cherry_(including_sour)___healthy       0.81      0.97      0.88      1102
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot       0.92      0.97      0.94      1102
                       Corn_(maize)___Common_rust_       0.96      0.96      0.96      1101
               Corn_(maize)___Northern_Leaf_Blight       0.90      0.93      0.